In [1]:
import sys
import time
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from langchain_groq import ChatGroq
from src.rag_pipeline import (
    load_hybrid_retriever,
    answer_query_hybrid,
    clean_response
)

In [2]:
# initiate LLM endpoint and model
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.2, #adjust higher for more creative responses,
    max_tokens=3000,
    )

response = llm.invoke("What is a washing machine?")
print(clean_response(response.content))

A **washing machine** is a household appliance designed to clean clothes, linens, and other fabrics automatically, replacing the manual labor of hand-washing. It uses water, detergent, and mechanical action (such as agitation or spinning) to remove dirt, stains, and odors from fabrics. Here's a breakdown of its key aspects:

### **How It Works**
1. **Loading**: Clothes are placed into a drum (the inner chamber).
2. **Water and Detergent**: The machine fills with water and adds detergent (manual or automatic).
3. **Washing Cycle**: 
   - **Agitation/Spin**: In top-loaders, an agitator or impeller moves clothes in the water. Front-loaders spin the drum to create motion.
   - **Rinsing**: Water is drained, and fresh water is added to rinse out detergent.
4. **Spinning**: The drum spins rapidly to remove excess water, reducing drying time.
5. **Draining**: Water is pumped out via a drain hose.

### **Types of Washing Machines**
- **Top-Loading**: Drum is accessed from the top; often uses a

In [3]:
# load processed data and indices for retrieval
data_path = root_dir / "data" / "processed" / "processed.parquet"
df = pd.read_parquet(data_path)
df_by_asin = df.set_index("parent_asin", drop=False)

print(f"Loaded {len(df)} products")

Loaded 20000 products


In [4]:
# load hybrid retriever
hybrid_retriever = load_hybrid_retriever()

print("Hybrid retriever loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hybrid retriever loaded successfully


In [5]:
# test hybrid retrieval alone without LLM
query = "energy efficient washing machine with quiet operation"
results = hybrid_retriever.retrieve(query, top_k=5)

print(f"Query: '{query}'")
print(f"\nTop {len(results)} results:")
for rank, (asin, score) in enumerate(results, start=1):
    title = df_by_asin.loc[asin].get("product_title", "Unknown Title") if asin in df_by_asin.index else "Unknown Title"
    print(f" {rank}. [{asin}] {title[:60]} | RRF Score: {score:.4f}")

Query: 'energy efficient washing machine with quiet operation'

Top 5 results:
 1. [B08R74X48Q] HOTSTORE Mini Washing Machine W/Spin Dryer, Electric Compact | RRF Score: 0.0142
 2. [B08XZJDMP4] Oneconcept Ecowash Pico – Portable Washing Machine, Compact  | RRF Score: 0.0083
 3. [B09TT5HQH8] Mini Washing Machine - 18 W of power Portable Ultrasonic Tur | RRF Score: 0.0083
 4. [B0B4P6G5QR] Portable Clothes Washing Machine,Ozone Sterilization Mini Wa | RRF Score: 0.0082
 5. [B08F4YGV8R] [SOPHIE MCCARTHY] 2020 NEW Versions Ultrasonic Turbine Steri | RRF Score: 0.0082


In [6]:
# test hybrid RAG pipeline with LLM response generation
answer, results = answer_query_hybrid(
    query,
    df_by_asin=df_by_asin,
    hybrid_retriever=hybrid_retriever,
    llm=llm,
)

print(f"Query: '{query}'")
print(f"\nAnswer:\n{answer}")

Query: 'energy efficient washing machine with quiet operation'

Answer:
The **Portable Clothes Washing Machine** (ASIN: B0B4P6G5QR) is described as having "quick and quiet operation" and includes ozone sterilization, making it a compact, energy-efficient option. While not explicitly labeled as "energy-saving," its mini design typically aligns with lower energy use. No other products in the context explicitly mention both energy efficiency and quiet operation.


In [7]:
# define evaluation queries (reused from the BM25 vs. semantic evaluation)
evaluation_queries = [
    "Washing machine",
    "Stainless steel coffee maker",
    "Replacement Parts DC61-02610A",
    "Magic bullet",
    "Whirlpool",
    "Washingmachine",
    "kitchen device to heat food quickly",
    "Something to cook pizza in",
    "Best appliances for a small apartment",
    "Aquamarine appliance to make bread crispy",
]

In [8]:
# run all evaluation queries through hybrid RAG pipeline and print results
for i, q in enumerate (evaluation_queries, start=1):
    print(f"{'='*60}")
    print(f"Query {i}: '{q}'")
    print(f"{'='*60}")

    answer, results = answer_query_hybrid(
        q,
        df_by_asin=df_by_asin,
        hybrid_retriever=hybrid_retriever,
        llm=llm,
    )

    print(f"\nAnswer:\n{answer}")
    print(f"\nTop retrieved products:")
    for rank, (asin, score) in enumerate(results, start=1):
        title = df_by_asin.loc[asin].get("product_title", "Unknown Title") if asin in df_by_asin.index else "Unknown Title"
        print(f" {rank}. {title[:60]} | RRF Score: {score:.4f}")
    print()

    time.sleep(15) # add delay to avoid hitting rate limits

Query 1: 'Washing machine'

Answer:
Based on the provided Amazon product data:

1. **Formemory Portable Folding Bucket Turbo Ultrasonic Washing Machine** (ASIN: B08FMND788)  
   - **Rating**: 5.0 (1 review, 2 helpful votes)  
   - **Key Features**: Compact, folds for storage, ideal for small/light loads (e.g., dog clothing).  

2. **Midea 1.6 CF Portable Washing Machine** (ASIN: B00DJF6296)  
   - **Rating**: 4.5 (8 reviews, 4 helpful votes)  
   - **Key Features**: Handles larger loads (queen sheets, jeans), but lacks portability without added wheels.  

For lightweight, travel-friendly needs: **B08FMND788** is highly recommended. For heavier use: **B00DJF6296** offers greater capacity. The USB-powered pink model (B09L1LHFP8) is listed at $17.89 but has no reviews.

Top retrieved products:
 1. Mini Washing Machine, Formemory Portable Folding Bucket Turb | RRF Score: 0.0165
 2. Mini Washing Machine, Mini Ultrasonic Washing Machine USB Po | RRF Score: 0.0145
 3. 4 PCS Washing Machine Fe